# AI Support Ticket Triage 

 
## Final Flow

```text
Subject + Body
      ↓
ML Model 1 → Issue Type
ML Model 2 → Priority
      ↓
Llama 3 via Ollama
      ↓
Queue + Summary + Main Problem + Recommended Action + Suggested Response
```

### Responsibility Split

**Machine Learning**
- Predict Issue Type
- Predict Priority

**Llama 3**
- Select the most appropriate Queue from a fixed allowed list
- Summarize the ticket
- Identify the main problem
- Recommend the next action
 

In [1]:
 # 2. IMPORTS
 
import json
import re
import requests
import pandas as pd
import joblib

In [18]:

# 3. CONFIGURATION
 
ISSUE_TYPE_MODEL_PATH = "issue_type_model.joblib"
PRIORITY_MODEL_PATH = "priority_model.joblib"

LLAMA_MODEL = "llama3.1"
OLLAMA_ENDPOINT = "http://localhost:11434/api/generate"

REQUEST_TIMEOUT = 120
TEMPERATURE = 0.2

# IMPORTANT:
ALLOWED_QUEUES = [
    "Billing and Payments",
    "Returns and Exchanges",
    "Technical Support",
    "Account Management",
    "General Inquiry"
]

## 4. Check Ollama Connection

In [3]:
# 4. CHECK OLLAMA


def check_ollama():
    try:
        response = requests.get(
            "http://localhost:11434/api/tags",
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        models = [
            model.get("name", "")
            for model in data.get("models", [])
        ]

        print("Ollama is running.")
        print("Available models:", models)

        if not any(
            model_name.startswith(LLAMA_MODEL)
            for model_name in models
        ):
            print(
                f"Warning: '{LLAMA_MODEL}' was not found. "
                f"Run: ollama pull {LLAMA_MODEL}"
            )

        return True

    except requests.RequestException as e:
        print("Ollama connection failed.")
        print("Make sure Ollama is running.")
        print("Error:", e)

        return False


check_ollama()

Ollama is running.
Available models: ['llama3:latest']


True

## 5. Load the Two ML Models

In [4]:
 # 5. LOAD TRAINED ML MODELS
 
issue_type_model = joblib.load(
    ISSUE_TYPE_MODEL_PATH
)

priority_model = joblib.load(
    PRIORITY_MODEL_PATH
)

print("Issue Type model loaded successfully.")
print("Priority model loaded successfully.")

Issue Type model loaded successfully.
Priority model loaded successfully.


## 6. Predict Issue Type and Priority



 

In [5]:
 # 6. ML PREDICTIONS
 
def predict_ticket_labels(subject, body):

    subject = "" if pd.isna(subject) else str(subject).strip()
    body = "" if pd.isna(body) else str(body).strip()

    ticket_text = f"{subject} {body}".strip()

    if not ticket_text:
        raise ValueError(
            "Ticket text cannot be empty."
        )

    predicted_type = issue_type_model.predict(
        [ticket_text]
    )[0]

    predicted_priority = priority_model.predict(
        [ticket_text]
    )[0]

    return {
        "ticket_text": ticket_text,
        "predicted_type": str(predicted_type),
        "predicted_priority": str(predicted_priority)
    }

## 7. Build the Llama Prompt

Llama receives:

- original subject
- original body
- ML-predicted Issue Type
- ML-predicted Priority
- fixed list of allowed Queues

Llama must choose exactly one Queue from that list.

In [6]:
 # 7. PROMPT BUILDER
 
def build_llama_prompt(
    subject,
    body,
    predicted_type,
    predicted_priority
):

    allowed_queues_text = "\n".join(
        f"- {queue}"
        for queue in ALLOWED_QUEUES
    )

    prompt = f"""
You are an AI assistant for a customer support ticket triage system.

The Machine Learning models have already classified the ticket's
Issue Type and Priority.

CUSTOMER TICKET

Subject:
{subject}

Body:
{body}


ML PREDICTIONS

Issue Type: {predicted_type}
Priority: {predicted_priority}


ALLOWED QUEUES

Choose exactly ONE Queue from the following list:

{allowed_queues_text}


RULES

1. Do not change the Issue Type.
2. Do not change the Priority.
3. Choose exactly one Queue from the allowed list.
4. Do not create a new Queue.
5. Select the Queue based on the ticket content and ML predictions.
6. Do not invent facts not present in the ticket.
7. Do not claim the issue has already been resolved.
8. Do not promise a resolution time unless explicitly stated.
9. Keep the output concise and professional.
10. Return valid JSON only.
11. Do not include markdown or explanation outside the JSON.

Return exactly this JSON structure:

{{
  "predicted_queue": "one exact Queue name from the allowed list",
  "summary": "a concise 1-2 sentence summary",
  "main_problem": "the main customer problem",
  "recommended_action": "the recommended next action for the selected support team",
  "suggested_response": "a short professional response to the customer"
}}
"""

    return prompt.strip()

## 8. Send the Prompt to Llama 3 Through Ollama

In [7]:
 # 8. LLAMA GENERATION THROUGH OLLAMA
 
def generate_with_llama(prompt):

    payload = {
        "model": LLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE
        }
    }

    response = requests.post(
        OLLAMA_ENDPOINT,
        json=payload,
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    result = response.json()

    if "response" not in result:
        raise ValueError(
            "Unexpected Ollama response format."
        )

    return result["response"]

## 9. Parse and Validate the Llama Output

This validation checks that:

- all required GenAI fields exist
- the returned Queue is one of the allowed Queue names

In [8]:
 # 9. JSON PARSER AND VALIDATOR
 
REQUIRED_GENAI_FIELDS = [
    "predicted_queue",
    "summary",
    "main_problem",
    "recommended_action",
    "suggested_response"
]


def parse_llama_json(raw_output):

    if not isinstance(raw_output, str):
        raise TypeError(
            "Llama output must be a string."
        )

    cleaned = raw_output.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if first_brace == -1 or last_brace == -1:
        raise ValueError(
            "No JSON object found in Llama output."
        )

    parsed = json.loads(
        cleaned[first_brace:last_brace + 1]
    )

    missing_fields = [
        field
        for field in REQUIRED_GENAI_FIELDS
        if field not in parsed
    ]

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )

    parsed_queue = str(
        parsed["predicted_queue"]
    ).strip()

    if parsed_queue not in ALLOWED_QUEUES:
        raise ValueError(
            f"Invalid queue returned by Llama: {parsed_queue}"
        )

    return {
        field: str(parsed[field]).strip()
        for field in REQUIRED_GENAI_FIELDS
    }

## 9b. Self-Correction Loop (Agent-like Retry)

Instead of calling Llama once and failing immediately if the JSON is invalid or the Queue is not in the allowed list, this step lets the pipeline **retry automatically**:

1. Call Llama and try to parse/validate the output.
2. If validation fails (bad JSON, missing field, invalid Queue), build a **correction prompt** that includes the exact error and asks Llama to fix it.
3. Retry up to `MAX_RETRIES` times.
4. If it still fails after all retries, raise the final error.

This does not require any external data or tools — it only uses information already available in the pipeline (the validation error itself), so it works fine with a synthetic/public dataset.

In [13]:
 # 9b. SELF-CORRECTION / RETRY LOOP

MAX_RETRIES = 3


def build_correction_prompt(original_prompt, bad_output, error_message):

    correction_prompt = f"""
{original_prompt}


YOUR PREVIOUS RESPONSE WAS INVALID.

Previous response:
{bad_output}

Validation error:
{error_message}

Fix the issue and return ONLY a valid JSON object that follows the exact structure and rules above. Do not include markdown or any text outside the JSON.
"""

    return correction_prompt.strip()


def generate_with_llama_and_validate(prompt, max_retries=MAX_RETRIES):

    current_prompt = prompt
    last_error = None

    for attempt in range(1, max_retries + 1):

        raw_output = generate_with_llama(current_prompt)

        try:
            genai_result = parse_llama_json(raw_output)

            if attempt > 1:
                print(
                    f"Llama output was corrected successfully "
                    f"on attempt {attempt}."
                )

            return genai_result

        except (ValueError, TypeError, json.JSONDecodeError) as e:

            last_error = e

            print(
                f"Attempt {attempt} failed validation: {e}"
            )

            if attempt < max_retries:
                current_prompt = build_correction_prompt(
                    original_prompt=prompt,
                    bad_output=raw_output,
                    error_message=str(e)
                )

    raise ValueError(
        f"Llama failed to produce valid output after "
        f"{max_retries} attempts. Last error: {last_error}"
    )

## 10. Complete End-to-End Pipeline

In [9]:
 # 10. COMPLETE PIPELINE
 
def process_ticket(subject, body):

    # Step 1: ML predictions
    ml_result = predict_ticket_labels(
        subject,
        body
    )

    # Step 2: Build Llama prompt
    prompt = build_llama_prompt(
        subject=subject,
        body=body,
        predicted_type=ml_result[
            "predicted_type"
        ],
        predicted_priority=ml_result[
            "predicted_priority"
        ]
    )

    # Step 3: Llama generation
    raw_llama_output = generate_with_llama(
        prompt
    )

    # Step 4: Parse and validate output
    genai_result = parse_llama_json(
        raw_llama_output
    )

    # Step 5: Final combined result
    return {
        "subject": subject,
        "body": body,
        "predicted_type": ml_result[
            "predicted_type"
        ],
        "predicted_priority": ml_result[
            "predicted_priority"
        ],
        **genai_result
    }

## 11. Test the ML Part Only

 

In [14]:
 # 11. TEST ML ONLY
 
example_subject = "Incorrect invoice amount"

example_body = (
    "Urgent The amount shown on my latest invoice is Wrong. "
    "Please review the charges."
)

ml_result = predict_ticket_labels(
    example_subject,
    example_body
)

print(
    json.dumps(
        ml_result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "ticket_text": "Incorrect invoice amount Urgent The amount shown on my latest invoice is Wrong. Please review the charges.",
  "predicted_type": "problem",
  "predicted_priority": "low"
}


In [15]:
test_tickets = [
    "Urgent critical system outage. All services are down.",
    "Severe security incident. Unauthorized access detected.",
    "Please update my account information.",
    "I have a small question about my invoice.",
    "Password reset request.",
]

for ticket in test_tickets:
    pred = priority_model.predict([ticket])[0]
    print(f"{pred:8} | {ticket}")

high     | Urgent critical system outage. All services are down.
high     | Severe security incident. Unauthorized access detected.
high     | Please update my account information.
medium   | I have a small question about my invoice.
high     | Password reset request.


## 12. Test the Complete ML + GenAI System

Run this after confirming Ollama is running.

In [16]:
#  12. COMPLETE EXAMPLE



result = process_ticket(
    example_subject,
    example_body
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "subject": "Incorrect invoice amount",
  "body": "Urgent The amount shown on my latest invoice is Wrong. Please review the charges.",
  "predicted_type": "problem",
  "predicted_priority": "low",
  "predicted_queue": "Billing and Payments",
  "summary": "Invoice amount discrepancy reported.",
  "main_problem": "Incorrect invoice amount displayed.",
  "recommended_action": "Verify invoice charges and investigate cause of discrepancy.",
  "suggested_response": "Thank you for bringing this to our attention. We will review your latest invoice and get back to you shortly."
}


## 13. True Agentic Layer: Tool Calling + Multi-Step Reasoning

Everything so far was a **single LLM call** (plus a retry loop if the JSON was invalid). This section turns Llama into an actual **agent**:

- It gets access to real **tools** (Python functions) it can call *during* generation.
- It can take **more than one step**: call a tool, look at the result, decide whether it needs another tool, and only then produce the final answer.
- All tools work **only on data we already have** (the ticket text and our own trained ML models) — no external APIs, no customer database, no knowledge base. This keeps everything compatible with a synthetic/public dataset.

**Note:** Tool calling requires a model that supports it in Ollama's `/api/chat` endpoint (e.g. `llama3.1`, `llama3.2`, `qwen2.5`, `mistral`). Plain `llama3` (3.0) does **not** reliably support tool calling. The next cell checks this automatically.

In [19]:
 # 14. CHECK TOOL-CALLING SUPPORT

OLLAMA_CHAT_ENDPOINT = "http://localhost:11434/api/chat"


def check_tool_calling_support(model=LLAMA_MODEL):

    test_tool = {
        "type": "function",
        "function": {
            "name": "ping",
            "description": "A test tool. Call it to say hello.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": "Call the ping tool now."
            }
        ],
        "tools": [test_tool],
        "stream": False
    }

    try:
        response = requests.post(
            OLLAMA_CHAT_ENDPOINT,
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
        data = response.json()

        tool_calls = data.get("message", {}).get("tool_calls")

        if tool_calls:
            print(f"'{model}' supports tool calling. Tool call detected:")
            print(tool_calls)
            return True
        else:
            print(
                f"'{model}' did NOT return a tool call. "
                f"It probably does not support tool calling reliably.\n"
                f"Consider running: ollama pull llama3.1\n"
                f"Model response: {data.get('message', {}).get('content', '')}"
            )
            return False

    except requests.RequestException as e:
        print("Could not reach Ollama /api/chat:", e)
        return False


TOOL_CALLING_SUPPORTED = check_tool_calling_support()

'llama3.1' supports tool calling. Tool call detected:
[{'id': 'call_u39jngy5', 'function': {'index': 0, 'name': 'ping', 'arguments': {'A': 'test'}}}]


## 15. Define the Agent's Tools

Four tools, all operating only on the ticket text and our own trained models:

1. **`get_ml_confidence`** — how confident the ML models actually are in their Issue Type / Priority prediction (uses `predict_proba`).
2. **`analyze_urgency_signals`** — rule-based scan of the ticket text for urgency-indicating language.
3. **`extract_ticket_entities`** — pulls out things like invoice/order numbers, dates, and emails mentioned in the ticket.
4. **`decide_escalation`** — combines priority, confidence, and urgency signals into an escalation recommendation.

In [20]:
 # 15. AGENT TOOLS (operate only on ticket text + our own ML models)

def get_ml_confidence(subject: str, body: str) -> dict:
    """Return how confident the ML models are in their predictions."""

    ticket_text = f"{subject} {body}".strip()
    result = {}

    for name, model in [
        ("issue_type", issue_type_model),
        ("priority", priority_model)
    ]:
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba([ticket_text])[0]
            classes = model.classes_
            best_idx = probs.argmax()
            result[name] = {
                "predicted_label": str(classes[best_idx]),
                "confidence": round(float(probs[best_idx]), 3)
            }
        else:
            result[name] = {
                "predicted_label": str(model.predict([ticket_text])[0]),
                "confidence": None
            }

    return result


URGENCY_KEYWORDS = [
    "urgent", "asap", "immediately", "critical", "emergency",
    "down", "outage", "security breach", "unauthorized",
    "can't access", "cannot access", "data loss", "broken"
]


def analyze_urgency_signals(subject: str, body: str) -> dict:
    """Scan ticket text for urgency-indicating keywords."""

    text = f"{subject} {body}".lower()

    matched = [
        kw for kw in URGENCY_KEYWORDS
        if kw in text
    ]

    return {
        "urgency_score": len(matched),
        "matched_keywords": matched
    }


def extract_ticket_entities(subject: str, body: str) -> dict:
    """Extract simple entities (invoice/order numbers, dates, emails) from the ticket."""

    text = f"{subject} {body}"

    invoice_or_order_numbers = re.findall(
        r"\b(?:INV|ORD|REF)?-?\d{4,}\b", text
    )
    emails = re.findall(
        r"[\w\.-]+@[\w\.-]+\.\w+", text
    )
    dates = re.findall(
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", text
    )

    return {
        "reference_numbers": invoice_or_order_numbers,
        "emails": emails,
        "dates": dates
    }


def decide_escalation(priority: str, urgency_score: int, ml_confidence: float) -> dict:
    """Decide whether the ticket should be escalated, based on our own signals."""

    should_escalate = (
        priority.lower() == "high"
        and (urgency_score >= 2 or (ml_confidence is not None and ml_confidence < 0.5))
    )

    reason = (
        "High priority combined with strong urgency signals or low ML confidence."
        if should_escalate
        else "No strong combined signal for escalation."
    )

    return {
        "should_escalate": should_escalate,
        "reason": reason
    }


TOOL_REGISTRY = {
    "get_ml_confidence": get_ml_confidence,
    "analyze_urgency_signals": analyze_urgency_signals,
    "extract_ticket_entities": extract_ticket_entities,
    "decide_escalation": decide_escalation
}

## 16. Tool Schemas (What Llama Sees)

This is the JSON-schema description of each tool that gets sent to Ollama so the model knows what tools exist, what they do, and what arguments they need.

In [21]:
 # 16. TOOL SCHEMAS (Ollama / OpenAI-style function definitions)

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_ml_confidence",
            "description": (
                "Get the ML models' confidence scores for Issue Type and "
                "Priority predictions on this ticket."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string"},
                    "body": {"type": "string"}
                },
                "required": ["subject", "body"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_urgency_signals",
            "description": (
                "Scan the ticket text for urgency-indicating keywords "
                "(e.g. urgent, outage, critical) and return a score."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string"},
                    "body": {"type": "string"}
                },
                "required": ["subject", "body"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_ticket_entities",
            "description": (
                "Extract reference numbers, emails, and dates mentioned in "
                "the ticket text."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string"},
                    "body": {"type": "string"}
                },
                "required": ["subject", "body"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "decide_escalation",
            "description": (
                "Decide whether this ticket should be escalated, based on "
                "priority, urgency score, and ML confidence. Call this AFTER "
                "you have urgency and confidence results."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "priority": {"type": "string"},
                    "urgency_score": {"type": "integer"},
                    "ml_confidence": {"type": "number"}
                },
                "required": ["priority", "urgency_score", "ml_confidence"]
            }
        }
    }
]

## 17. The Agent Loop (Multi-Step Reasoning + Tool Calling)

This is the actual agent: instead of one fixed prompt, Llama gets a **system prompt + the ticket**, and then runs in a loop:

1. Llama decides: *do I need a tool, or can I answer now?*
2. If it asks for a tool, we run the real Python function and feed the result back to it.
3. It can call more tools based on what it learned (multi-step).
4. Once it has enough information, it produces the final JSON, which we validate (reusing the self-correction loop from Section 9b logic).

A `MAX_AGENT_STEPS` limit prevents infinite loops.

In [22]:
 # 17. AGENT ORCHESTRATOR

MAX_AGENT_STEPS = 6


def build_agent_system_prompt(predicted_type, predicted_priority):

    allowed_queues_text = "\n".join(
        f"- {queue}" for queue in ALLOWED_QUEUES
    )

    return f"""
You are an autonomous AI agent for a customer support ticket triage system.

You have access to tools that inspect the ticket and our ML models. Use them 
whenever they would help you decide better, especially before recommending 
escalation. You may call multiple tools, one at a time, before answering.

ML PREDICTIONS (already computed, do not change them):
Issue Type: {predicted_type}
Priority: {predicted_priority}

ALLOWED QUEUES (choose exactly one):
{allowed_queues_text}

When you are done gathering information, respond with ONLY this JSON object 
and nothing else (no markdown, no explanation):

{{
  "predicted_queue": "one exact Queue name from the allowed list",
  "summary": "a concise 1-2 sentence summary",
  "main_problem": "the main customer problem",
  "recommended_action": "the recommended next action for the selected support team",
  "suggested_response": "a short professional response to the customer",
  "escalate": true or false
}}
""".strip()


def call_ollama_chat(messages, tools=None):

    payload = {
        "model": LLAMA_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": TEMPERATURE}
    }

    if tools:
        payload["tools"] = tools

    response = requests.post(
        OLLAMA_CHAT_ENDPOINT,
        json=payload,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()

    return response.json()["message"]


def run_agentic_pipeline(subject, body):

    # Step 0: ML predictions, exactly like the non-agentic pipeline
    ml_result = predict_ticket_labels(subject, body)

    system_prompt = build_agent_system_prompt(
        predicted_type=ml_result["predicted_type"],
        predicted_priority=ml_result["predicted_priority"]
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Subject: {subject}\n\nBody: {body}"
        }
    ]

    final_raw_output = None

    for step in range(1, MAX_AGENT_STEPS + 1):

        assistant_message = call_ollama_chat(
            messages,
            tools=TOOL_SCHEMAS if TOOL_CALLING_SUPPORTED else None
        )

        tool_calls = assistant_message.get("tool_calls")

        if tool_calls:

            messages.append(assistant_message)

            for call in tool_calls:

                tool_name = call["function"]["name"]
                tool_args = call["function"].get("arguments", {})

                print(f"Step {step}: agent is calling tool '{tool_name}' with {tool_args}")

                if tool_name not in TOOL_REGISTRY:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}
                else:
                    try:
                        tool_result = TOOL_REGISTRY[tool_name](**tool_args)
                    except Exception as e:
                        tool_result = {"error": str(e)}

                messages.append({
                    "role": "tool",
                    "content": json.dumps(tool_result, ensure_ascii=False)
                })

            continue

        # No tool call -> this should be the final answer
        final_raw_output = assistant_message.get("content", "")
        break

    else:
        raise RuntimeError(
            f"Agent exceeded {MAX_AGENT_STEPS} steps without a final answer."
        )

    # Validate the final JSON, with the same self-correction retry logic
    try:
        genai_result = parse_llama_json(final_raw_output)
        genai_result["escalate"] = json.loads(final_raw_output).get("escalate", False)
    except (ValueError, TypeError, json.JSONDecodeError) as e:
        print(f"Final answer failed validation ({e}), asking agent to fix it...")
        correction_prompt = build_correction_prompt(
            original_prompt=system_prompt,
            bad_output=final_raw_output,
            error_message=str(e)
        )
        messages.append({"role": "user", "content": correction_prompt})
        fixed_message = call_ollama_chat(messages)
        genai_result = parse_llama_json(fixed_message.get("content", ""))
        genai_result["escalate"] = json.loads(fixed_message.get("content", "{}")).get("escalate", False)

    return {
        "subject": subject,
        "body": body,
        "predicted_type": ml_result["predicted_type"],
        "predicted_priority": ml_result["predicted_priority"],
        **genai_result
    }

## 18. Test the Full Agentic System

In [23]:
 # 18. TEST AGENTIC PIPELINE

agentic_result = run_agentic_pipeline(
    example_subject,
    example_body
)

print(
    json.dumps(
        agentic_result,
        indent=2,
        ensure_ascii=False
    )
)

Step 1: agent is calling tool 'analyze_urgency_signals' with {'subject': 'Incorrect invoice amount', 'body': 'Urgent The amount shown on my latest invoice is Wrong. Please review the charges.'}
{
  "subject": "Incorrect invoice amount",
  "body": "Urgent The amount shown on my latest invoice is Wrong. Please review the charges.",
  "predicted_type": "problem",
  "predicted_priority": "low",
  "predicted_queue": "Billing and Payments",
  "summary": "Customer reports incorrect invoice amount, requesting review of charges.",
  "main_problem": "Invoice amount discrepancy",
  "recommended_action": "Verify customer's account information and recent transactions to identify the cause of the discrepancy.",
  "suggested_response": "Thank you for bringing this to our attention. We will review your latest invoice and investigate the issue. Please allow 24-48 hours for us to look into this further.",
  "escalate": false
}
